In [34]:
import pandas as pd
import numpy as np
import glob
import os
from tqdm import tqdm

In [35]:
!pwd

/storage/Arushi/090526_EvoAge/kg_formation/processed_data_relation_wise_merge/generalised/OTHER_SPECIES/Mouse


In [36]:
BASE_DIR     = '/storage/Arushi/090526_EvoAge/kg_formation/'
MAPPING_DIR  = BASE_DIR + 'data_collection/databases_for_mapping/'
PROC_DIR     = BASE_DIR + 'processed_data/'

# ── Output path ───────────────────────────────────────────────────────────────
!mkdir Mouse_phenotype_biologicalprocess
OUT_PATH = BASE_DIR + 'processed_data_relation_wise_merge/generalised/OTHER_SPECIES/Mouse/Mouse_phenotype_biologicalprocess/Mouse_phenotype_biologicalprocess.csv'

# ── Required output schema ────────────────────────────────────────────────────
REQUIRED_COLS = [
    'head', 'relation', 'tail',
    'head_type', 'relation_type', 'tail_type',
    'kg_source', 'kg_type',
    'head_id_is', 'tail_id_is',
    'head_detail_name', 'tail_detail_name', 'species'
]

mkdir: cannot create directory ‘Mouse_phenotype_biologicalprocess’: File exists


In [37]:
#

In [38]:
# Mouse = pd.read_csv(
#     f'{BASE_DIR}data_collection/databases_for_mapping/ncbi/Mus_musculus.gene_info',
#     sep='\t',
    
# )
# # Mouse["LocusTag"] = Mouse["LocusTag"].str.replace("Dmel_", "", regex=False)

# # Mouse_symbol2Locus_dict = dict(zip(Mouse['Symbol'], Mouse['LocusTag']))
# # Mouse
# # Mouse_symbol2Locus_dict
# Mouse

# monarch

In [39]:
monarch = pd.read_csv(PROC_DIR + 'Monarch/Monarch_final/Mouse/Mouse_PhenotypicFeature_BiologicalProcess.csv')

monarch.columns = monarch.columns.str.lower()
monarch = monarch.rename(columns={
    'headtype': 'head_type',
    'tailtype': 'tail_type',
})
# monarch = monarch[~monarch['head_detail_name'].isna()]
monarch['kg_type'] = 'Generalised'
monarch['kg_source'] = 'Monarch_KG'
monarch['species'] = 'M.musculus'
monarch['head_id_is'] = 'MP_ID'
print(f"monarch: {len(monarch):,} rows")
monarch

monarch: 1,493 rows


,head,relation,tail,head_type,relation_type,tail_type,relation_source,head_detail_name,tail_detail_name,head_taxon_name,tail_taxon_name,head_id_is,tail_id_is,species,kg_type,kg_source
0,MP:0000015,Phenotype_BiologicalProcess,GO:0043473,Phenotype,related_to,BiologicalProcess,infores:upheno,abnormal ear pigmentation,pigmentation,Mus musculus,NaN,MP_ID,Quick_GO,M.musculus,Generalised,Monarch_KG
1,MP:0000052,Phenotype_BiologicalProcess,GO:0060185,Phenotype,related_to,BiologicalProcess,infores:upheno,early ear unfolding,outer ear unfolding,Mus musculus,NaN,MP_ID,Quick_GO,M.musculus,Generalised,Monarch_KG
2,MP:0000054,Phenotype_BiologicalProcess,GO:0060186,Phenotype,related_to,BiologicalProcess,infores:upheno,delayed ear emergence,outer ear emergence,Mus musculus,NaN,MP_ID,Quick_GO,M.musculus,Generalised,Monarch_KG
3,MP:0000060,Phenotype_BiologicalProcess,GO:0001503,Phenotype,related_to,BiologicalProcess,infores:upheno,delayed bone ossification,ossification,Mus musculus,NaN,MP_ID,Quick_GO,M.musculus,Generalised,Monarch_KG
4,MP:0000064,Phenotype_BiologicalProcess,GO:0045453,Phenotype,related_to,BiologicalProcess,infores:upheno,failure of bone resorption,bone resorption,Mus musculus,NaN,MP_ID,Quick_GO,M.musculus,Generalised,Monarch_KG
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1488,MP:0031633,Phenotype_BiologicalProcess,GO:0060048,Phenotype,related_to,BiologicalProcess,infores:upheno,decreased heart left ventricle muscle contract...,cardiac muscle contraction,Mus musculus,NaN,MP_ID,Quick_GO,M.musculus,Generalised,Monarch_KG
1489,MP:0031634,Phenotype_BiologicalProcess,GO:0060048,Phenotype,related_to,BiologicalProcess,infores:upheno,decreased heart right ventricle muscle contrac...,cardiac muscle contraction,Mus musculus,NaN,MP_ID,Quick_GO,M.musculus,Generalised,Monarch_KG
1490,MP:0031635,Phenotype_BiologicalProcess,GO:0060048,Phenotype,related_to,BiologicalProcess,infores:upheno,increased heart left ventricle muscle contract...,cardiac muscle contraction,Mus musculus,NaN,MP_ID,Quick_GO,M.musculus,Generalised,Monarch_KG
1491,MP:0031636,Phenotype_BiologicalProcess,GO:0060048,Phenotype,related_to,BiologicalProcess,infores:upheno,increased heart right ventricle muscle contrac...,cardiac muscle contraction,Mus musculus,NaN,MP_ID,Quick_GO,M.musculus,Generalised,Monarch_KG


# Consolidate into Unified Schem

In [40]:
# List all source DataFrames to include
source_dfs = [
    monarch,
]

aligned = []
for df in source_dfs:
    df = df.copy()
    for col in REQUIRED_COLS:
        if col not in df.columns:
            df[col] = None       
    aligned.append(df[REQUIRED_COLS])

final_df = pd.concat(aligned, ignore_index=True)
print(f"Consolidated rows: {len(final_df):,}")
final_df

Consolidated rows: 1,493


,head,relation,tail,head_type,relation_type,tail_type,kg_source,kg_type,head_id_is,tail_id_is,head_detail_name,tail_detail_name,species
0,MP:0000015,Phenotype_BiologicalProcess,GO:0043473,Phenotype,related_to,BiologicalProcess,Monarch_KG,Generalised,MP_ID,Quick_GO,abnormal ear pigmentation,pigmentation,M.musculus
1,MP:0000052,Phenotype_BiologicalProcess,GO:0060185,Phenotype,related_to,BiologicalProcess,Monarch_KG,Generalised,MP_ID,Quick_GO,early ear unfolding,outer ear unfolding,M.musculus
2,MP:0000054,Phenotype_BiologicalProcess,GO:0060186,Phenotype,related_to,BiologicalProcess,Monarch_KG,Generalised,MP_ID,Quick_GO,delayed ear emergence,outer ear emergence,M.musculus
3,MP:0000060,Phenotype_BiologicalProcess,GO:0001503,Phenotype,related_to,BiologicalProcess,Monarch_KG,Generalised,MP_ID,Quick_GO,delayed bone ossification,ossification,M.musculus
4,MP:0000064,Phenotype_BiologicalProcess,GO:0045453,Phenotype,related_to,BiologicalProcess,Monarch_KG,Generalised,MP_ID,Quick_GO,failure of bone resorption,bone resorption,M.musculus
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1488,MP:0031633,Phenotype_BiologicalProcess,GO:0060048,Phenotype,related_to,BiologicalProcess,Monarch_KG,Generalised,MP_ID,Quick_GO,decreased heart left ventricle muscle contract...,cardiac muscle contraction,M.musculus
1489,MP:0031634,Phenotype_BiologicalProcess,GO:0060048,Phenotype,related_to,BiologicalProcess,Monarch_KG,Generalised,MP_ID,Quick_GO,decreased heart right ventricle muscle contrac...,cardiac muscle contraction,M.musculus
1490,MP:0031635,Phenotype_BiologicalProcess,GO:0060048,Phenotype,related_to,BiologicalProcess,Monarch_KG,Generalised,MP_ID,Quick_GO,increased heart left ventricle muscle contract...,cardiac muscle contraction,M.musculus
1491,MP:0031636,Phenotype_BiologicalProcess,GO:0060048,Phenotype,related_to,BiologicalProcess,Monarch_KG,Generalised,MP_ID,Quick_GO,increased heart right ventricle muscle contrac...,cardiac muscle contraction,M.musculus


# Sanity Check — Distinct Values

In [41]:
for col in ['relation', 'head_type', 'tail_type', 'relation_type', 'kg_source', 'head_id_is', 'tail_id_is']:
    print(f"{col:20s}: {set(final_df[col])}")

relation            : {'Phenotype_BiologicalProcess'}
head_type           : {'Phenotype'}
tail_type           : {'BiologicalProcess'}
relation_type       : {'related_to'}
kg_source           : {'Monarch_KG'}
head_id_is          : {'MP_ID'}
tail_id_is          : {'Quick_GO'}


In [42]:
# Step 4: drop unresolvable rows
before = len(final_df)
final_df = final_df[~final_df['tail_detail_name'].isna()].reset_index(drop=True)
print(f"Dropped {before - len(final_df):,} unresolvable rows → {len(final_df):,} remaining")

Dropped 0 unresolvable rows → 1,493 remaining


# NaN Audit (pre-dedup)

In [43]:
true_nan   = final_df.isna().sum()
string_nan = final_df.apply(lambda col: col.astype(str).str.upper().eq('NAN').sum())

pd.DataFrame({
    'NaN_count':          true_nan,
    "'NAN'_string_count": string_nan,
    'Total_NaN_like':     true_nan + string_nan
})

,NaN_count,'NAN'_string_count,Total_NaN_like
head,0,0,0
relation,0,0,0
tail,0,0,0
head_type,0,0,0
relation_type,0,0,0
tail_type,0,0,0
kg_source,0,0,0
kg_type,0,0,0
head_id_is,0,0,0
tail_id_is,0,0,0


# Deduplication

In [44]:
def merge_sources(x):
    """Combine unique, non-null source labels with '::' delimiter."""
    return '::'.join(sorted(set(x.dropna())))

group_cols = ['head', 'relation', 'tail']

final_df_dedup = final_df.groupby(group_cols, as_index=False).agg({
    'head_type':        'first',
    'relation_type':    'first',
    'tail_type':        'first',
    'kg_source':        merge_sources,
    'kg_type':          merge_sources,   # ← changed from 'first'
    'head_id_is':       'first',
    'tail_id_is':       'first',
    'head_detail_name': 'first',
    'tail_detail_name': 'first',
    'species': 'first'
})

print(f"Before dedup: {len(final_df):,}  |  After dedup: {len(final_df_dedup):,}")
final_df_dedup.head(3)

Before dedup: 1,493  |  After dedup: 1,493


,head,relation,tail,head_type,relation_type,tail_type,kg_source,kg_type,head_id_is,tail_id_is,head_detail_name,tail_detail_name,species
0,MP:0000015,Phenotype_BiologicalProcess,GO:0043473,Phenotype,related_to,BiologicalProcess,Monarch_KG,Generalised,MP_ID,Quick_GO,abnormal ear pigmentation,pigmentation,M.musculus
1,MP:0000052,Phenotype_BiologicalProcess,GO:0060185,Phenotype,related_to,BiologicalProcess,Monarch_KG,Generalised,MP_ID,Quick_GO,early ear unfolding,outer ear unfolding,M.musculus
2,MP:0000054,Phenotype_BiologicalProcess,GO:0060186,Phenotype,related_to,BiologicalProcess,Monarch_KG,Generalised,MP_ID,Quick_GO,delayed ear emergence,outer ear emergence,M.musculus


In [45]:
true_nan   = final_df_dedup.isna().sum()
string_nan = final_df_dedup.apply(lambda col: col.astype(str).str.upper().eq('NAN').sum())

pd.DataFrame({
    'NaN_count':          true_nan,
    "'NAN'_string_count": string_nan,
    'Total_NaN_like':     true_nan + string_nan
})

,NaN_count,'NAN'_string_count,Total_NaN_like
head,0,0,0
relation,0,0,0
tail,0,0,0
head_type,0,0,0
relation_type,0,0,0
tail_type,0,0,0
kg_source,0,0,0
kg_type,0,0,0
head_id_is,0,0,0
tail_id_is,0,0,0


In [46]:
print("kg_source values present:", set(final_df_dedup['kg_source']), final_df_dedup['kg_source'].value_counts())

kg_source values present: {'Monarch_KG'} kg_source
Monarch_KG    1493
Name: count, dtype: int64


In [47]:
print("kg_source values present:", set(final_df_dedup['kg_type']), final_df_dedup['kg_type'].value_counts())

kg_source values present: {'Generalised'} kg_type
Generalised    1493
Name: count, dtype: int64


In [48]:
final_df_dedup.to_csv(OUT_PATH, index=False)
print(f"Saved {len(final_df_dedup):,} rows → {OUT_PATH}")

Saved 1,493 rows → /storage/Arushi/090526_EvoAge/kg_formation/processed_data_relation_wise_merge/generalised/OTHER_SPECIES/Mouse/Mouse_phenotype_biologicalprocess/Mouse_phenotype_biologicalprocess.csv


In [49]:
# OUT_PATH